#**Healthcare Insurance Cost Analysis Project:**#

This project set out to explore how Data Analytics and AI tools can be use to extract inside information from raw datasets into meaningful insights that utilise for prediction and decesion manking in data driven organisation and for business success. The healthcare Insurance Cost Analysis Project seeks empirically to explore systematic approach to analyse raw datasets and its rationale to identify how different age, gender, habit and health condition affect health insureance cost in various location. In this Jupyter notebook, therefore, performing the steps: Section 1.0 to 1. data cleaning and Section 2.0 to 2.  data visualisation. 

*  For this analysis fetch data from Kaggle and save to local drive. Data then clean and processed for visualisation
*  Python programming language and its libraries are used in this analysis. Pyhton version use 3.12.8 and Libraries use - Numpy, Pandas, Matplotlib, Seaborn, plotly.

---

*Setting up working environment and directory*

Access the current directory with os.getcwd()

In [60]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users'

Make the parent of the current directory the new current directory with os.path.dirname() to get the parent directory and os.chir() defines the new current directory

In [61]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [62]:
current_dir = os.getcwd()
current_dir

'c:\\'

# Section 1

**Import, Clean, Transform and save as a clean dataset for visualisation and further analysis**

**1.0 Import Python libraries and Dataset**

Import pandas library which run on top of numerical python: Numpy backed by core python terminal. To import data file from souce can be performed in various ways by defining the file path on pandas data frame. Since, this project data already downloaded from Kaggle.com and store in local computer, local file path will use. However, direct source link can be used if required by providing full file path. For example- Local path: C:\\Users\\Documents\\insurance.csv
Or source path: https://www.kaggle.com/datasets/willianoliveiragibin/healthcare-insurance?select=insurance.csv

In [63]:
import pandas as pd
import numpy as np

Import Data to pandas data frame and get an initial overview of data summary to generate thought process. Visualise some rows and columns. Below code will visualise 20 rows from source file.

In [64]:
insurance_raw = pd.read_csv("C:\\Users\\skmra\\Documents\\healthcare_insurance_cost_analysis\\insurance.csv")

print(insurance_raw.shape)   # (891, 8)
insurance_raw.head(20)

(1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
6,46,female,33.440,1,no,southeast,8240.58960
7,37,female,27.740,3,no,northwest,7281.50560
8,37,male,29.830,2,no,northeast,6406.41070
9,60,female,25.840,0,no,northwest,28923.13692


**1.1 Check Summary of initial Dataset and find missing values**

To understand the dataset initially few steps are considers. First, check what type of data in the cell, any missing data or any empty cells, verify this by checking with non empty cell to match with with total cell. Also check any unque values that is not very common in the datasets. 

In [65]:
before_summary = pd.DataFrame({
    "dtype": insurance_raw.dtypes,
    "missing_values": insurance_raw.isnull().sum(),
    "non_missing": insurance_raw.notnull().sum(),
    "unique_values": insurance_raw.nunique()
})
before_summary

,dtype,missing_values,non_missing,unique_values
age,int64,0,1338,47
sex,str,0,1338,2
bmi,float64,0,1338,548
children,int64,0,1338,6
smoker,str,0,1338,2
region,str,0,1338,4
charges,float64,0,1338,1337


After checking the datasets no missing values are found. However, if missing values are found then pandas missing values handling techniques will use.

**1.2 Check for duplicate data, Rename all the columns and create new columns**

Make a copy of the DataFrame and check for duplicate if found it will delete from the data frame by using drop_duplicates(inplace=True)

In [66]:
insurance = insurance_raw.copy()
insurance.drop_duplicates(inplace=True)


Rename all the columns names to title case by using python for loop statesment. Change Bmi columns to B_M_I

In [67]:
insurance.columns = [col.title() for col in insurance.columns]
insurance.rename(columns={'Bmi': 'B_M_I'}, inplace=True)
insurance.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Age       1337 non-null   int64  
 1   Sex       1337 non-null   str    
 2   B_M_I     1337 non-null   float64
 3   Children  1337 non-null   int64  
 4   Smoker    1337 non-null   str    
 5   Region    1337 non-null   str    
 6   Charges   1337 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 83.6 KB


view the unique values in each of the columns that will be categories:

In [68]:
categorical_columns = ['Sex', 'Smoker', 'Region']

for col in categorical_columns:
    print(f"Unique values in {col} are : {insurance[col].unique()}")

Unique values in Sex are : <StringArray>
['female', 'male']
Length: 2, dtype: str
Unique values in Smoker are : <StringArray>
['yes', 'no']
Length: 2, dtype: str
Unique values in Region are : <StringArray>
['southwest', 'southeast', 'northwest', 'northeast']
Length: 4, dtype: str


And arrange these category data type and view the updated column types:

In [69]:
categorical_columns = ['Sex', 'Smoker', 'Region']

for col in categorical_columns:
    insurance[col] = insurance[col].astype('category')
insurance.info()

<class 'pandas.DataFrame'>
Index: 1337 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   Age       1337 non-null   int64   
 1   Sex       1337 non-null   category
 2   B_M_I     1337 non-null   float64 
 3   Children  1337 non-null   int64   
 4   Smoker    1337 non-null   category
 5   Region    1337 non-null   category
 6   Charges   1337 non-null   float64 
dtypes: category(3), float64(2), int64(2)
memory usage: 56.2 KB


**1.3 In this section each columns data will be group and analyse.** 

1.3.0 Add new Columns for age group 

In [70]:
insurance['Age_Group'] = pd.cut(
    insurance['Age'],
    bins=[17, 26, 35, 45, 55, 100],
    labels=['Ages 18-25', 'Ages 26-34', 'Ages 35-44', 'Ages 45-54', 'Ages 55 and over'],
    right=False
)

insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group
0,19,female,27.900,0,yes,southwest,16884.92400,Ages 18-25
1,18,male,33.770,1,no,southeast,1725.55230,Ages 18-25
2,28,male,33.000,3,no,southeast,4449.46200,Ages 26-34
3,33,male,22.705,0,no,northwest,21984.47061,Ages 26-34
4,32,male,28.880,0,no,northwest,3866.85520,Ages 26-34


**1.3.1 Add new Columns for B_M_I group**

After researching online and publicly available information Age and B_M_I columns are group in the following ways. 
 age groups as age range as follows-

- `< 26` - 'Ages 18-25'
- `>= 26 < 35` - 'Ages 26-34'
- `>= 35 < 45` - 'Ages 35-44'
- `>= 44 < 55` - 'Ages 44-54'
- `>= 55` - 'Ages 55 and over'

and B_M_I group as follows

- `below 18.5` – you're in the underweight range
- `18.5 to 24.9` – you're in the healthy weight range
- `25 to 29.9` – you're in the overweight range
- `30 to 39.9` – you're in the obese range
- `40 or above` – you're in the severely obese range

Therefore, new group BMI_Group will be as follows-

- `< 18.5` - 'Underweight'
- `>= 18.5 < 25` - 'Healthy Weight'
- `>= 25 < 30` - 'Overweight'
- `>= 30 < 40` - 'Obese'
- `>= 40` - 'Severely Obese'

In [71]:
insurance['BMI_Group'] = pd.cut(
    insurance['B_M_I'],
    bins=[0, 18.5, 25, 30, 40, 100],
    labels=['Underweight', 'Healthy Weight', 'Overweight', 'Obese', 'Severely Obese'],
    right=False
)

insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group
0,19,female,27.900,0,yes,southwest,16884.92400,Ages 18-25,Overweight
1,18,male,33.770,1,no,southeast,1725.55230,Ages 18-25,Obese
2,28,male,33.000,3,no,southeast,4449.46200,Ages 26-34,Obese
3,33,male,22.705,0,no,northwest,21984.47061,Ages 26-34,Healthy Weight
4,32,male,28.880,0,no,northwest,3866.85520,Ages 26-34,Overweight


---

**1.3.2 Add new Columns for insurance cost plan**

For Futher analysis in the dataset, will add Plan column which will create based on the individual with children or no children. If individual have no children the plan will be standard. If individual have children then it will be a Family plan.

In [72]:
insurance['Plan'] = np.where(insurance['Children'] > 0, 'Family', 'Standard')
insurance['Plan'] = insurance['Plan'].astype('category')
insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group,Plan
0,19,female,27.900,0,yes,southwest,16884.92400,Ages 18-25,Overweight,Standard
1,18,male,33.770,1,no,southeast,1725.55230,Ages 18-25,Obese,Family
2,28,male,33.000,3,no,southeast,4449.46200,Ages 26-34,Obese,Family
3,33,male,22.705,0,no,northwest,21984.47061,Ages 26-34,Healthy Weight,Standard
4,32,male,28.880,0,no,northwest,3866.85520,Ages 26-34,Overweight,Standard


**1.3.3 Fromating the title case all the values have a capital letter first**

In [73]:
insurance['Sex'] = insurance['Sex'].cat.rename_categories(lambda x: x.title())
insurance['Smoker'] = insurance['Smoker'].cat.rename_categories(lambda x: x.title())
insurance['Region'] = insurance['Region'].cat.rename_categories(lambda x: x.title())
insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group,Plan
0,19,Female,27.900,0,Yes,Southwest,16884.92400,Ages 18-25,Overweight,Standard
1,18,Male,33.770,1,No,Southeast,1725.55230,Ages 18-25,Obese,Family
2,28,Male,33.000,3,No,Southeast,4449.46200,Ages 26-34,Obese,Family
3,33,Male,22.705,0,No,Northwest,21984.47061,Ages 26-34,Healthy Weight,Standard
4,32,Male,28.880,0,No,Northwest,3866.85520,Ages 26-34,Overweight,Standard


**1.3.4 Add a new columns to calculate per person cost**

By considering individuals and number of children on the insurance plan

In [74]:
insurance['Charges_Per_Person'] = insurance['Charges'] / (insurance['Children'] + 1)
insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group,Plan,Charges_Per_Person
0,19,Female,27.900,0,Yes,Southwest,16884.92400,Ages 18-25,Overweight,Standard,16884.92400
1,18,Male,33.770,1,No,Southeast,1725.55230,Ages 18-25,Obese,Family,862.77615
2,28,Male,33.000,3,No,Southeast,4449.46200,Ages 26-34,Obese,Family,1112.36550
3,33,Male,22.705,0,No,Northwest,21984.47061,Ages 26-34,Healthy Weight,Standard,21984.47061
4,32,Male,28.880,0,No,Northwest,3866.85520,Ages 26-34,Overweight,Standard,3866.85520


**1.3.4 Add outlier columns and numeric columns for categories** 


In [75]:
def check_outlier(s):
    """ Calculating inter quartile range and returning True or False"""
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    return (s < low) | (s > high)

insurance['Charges_Outlier'] = check_outlier(insurance['Charges'])
insurance['Charges_Per_Person_Outlier'] = check_outlier(insurance['Charges_Per_Person'])
insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group,Plan,Charges_Per_Person,Charges_Outlier,Charges_Per_Person_Outlier
0,19,Female,27.900,0,Yes,Southwest,16884.92400,Ages 18-25,Overweight,Standard,16884.92400,False,False
1,18,Male,33.770,1,No,Southeast,1725.55230,Ages 18-25,Obese,Family,862.77615,False,False
2,28,Male,33.000,3,No,Southeast,4449.46200,Ages 26-34,Obese,Family,1112.36550,False,False
3,33,Male,22.705,0,No,Northwest,21984.47061,Ages 26-34,Healthy Weight,Standard,21984.47061,False,False
4,32,Male,28.880,0,No,Northwest,3866.85520,Ages 26-34,Overweight,Standard,3866.85520,False,False


In [76]:
cat_cols = ['Sex', 'Smoker', 'Region', 'Plan', 'BMI_Group', 'Age_Group']
for col in cat_cols:
    insurance[f"{col}_num"] = insurance[col].cat.codes

insurance.head()

,Age,Sex,B_M_I,Children,Smoker,Region,Charges,Age_Group,BMI_Group,Plan,Charges_Per_Person,Charges_Outlier,Charges_Per_Person_Outlier,Sex_num,Smoker_num,Region_num,Plan_num,BMI_Group_num,Age_Group_num
0,19,Female,27.900,0,Yes,Southwest,16884.92400,Ages 18-25,Overweight,Standard,16884.92400,False,False,0,1,3,1,2,0
1,18,Male,33.770,1,No,Southeast,1725.55230,Ages 18-25,Obese,Family,862.77615,False,False,1,0,2,0,3,0
2,28,Male,33.000,3,No,Southeast,4449.46200,Ages 26-34,Obese,Family,1112.36550,False,False,1,0,2,0,3,1
3,33,Male,22.705,0,No,Northwest,21984.47061,Ages 26-34,Healthy Weight,Standard,21984.47061,False,False,1,0,1,1,1,1
4,32,Male,28.880,0,No,Northwest,3866.85520,Ages 26-34,Overweight,Standard,3866.85520,False,False,1,0,1,1,2,1


**1.3.5 Rearranging the columns to improve readability**

In [77]:
insurance = insurance[[
    'Age', 'Age_Group', 'Age_Group_num',
    'Sex', 'Sex_num',
    'B_M_I', 'BMI_Group', 'BMI_Group_num',
    'Children', 'Plan', 'Plan_num',
    'Smoker', 'Smoker_num',
    'Region', 'Region_num',
    'Charges', 'Charges_Per_Person',
    'Charges_Outlier', 'Charges_Per_Person_Outlier'
]]
insurance.head()

,Age,Age_Group,Age_Group_num,Sex,Sex_num,B_M_I,BMI_Group,BMI_Group_num,Children,Plan,Plan_num,Smoker,Smoker_num,Region,Region_num,Charges,Charges_Per_Person,Charges_Outlier,Charges_Per_Person_Outlier
0,19,Ages 18-25,0,Female,0,27.900,Overweight,2,0,Standard,1,Yes,1,Southwest,3,16884.92400,16884.92400,False,False
1,18,Ages 18-25,0,Male,1,33.770,Obese,3,1,Family,0,No,0,Southeast,2,1725.55230,862.77615,False,False
2,28,Ages 26-34,1,Male,1,33.000,Obese,3,3,Family,0,No,0,Southeast,2,4449.46200,1112.36550,False,False
3,33,Ages 26-34,1,Male,1,22.705,Healthy Weight,1,0,Standard,1,No,0,Northwest,1,21984.47061,21984.47061,False,False
4,32,Ages 26-34,1,Male,1,28.880,Overweight,2,0,Standard,1,No,0,Northwest,1,3866.85520,3866.85520,False,False


**1.4 Comparing raw data set with newly cleaned dataset**

**1.4.0 Create after summary of the data set**

In [80]:
after_summary = pd.DataFrame({
    "dtype": insurance.dtypes,
    "missing_values": insurance.isnull().sum(),
    "non_missing": insurance.notnull().sum(),
    "unique_values": insurance.nunique()
})
after_summary

,dtype,missing_values,non_missing,unique_values
Age,int64,0,1337,47
Age_Group,category,0,1337,5
Age_Group_num,int8,0,1337,5
Sex,category,0,1337,2
Sex_num,int8,0,1337,2
B_M_I,float64,0,1337,548
BMI_Group,category,0,1337,5
BMI_Group_num,int8,0,1337,5
Children,int64,0,1337,6
Plan,category,0,1337,2


**1.4.0 Create before summary of the data set for a view**

In [81]:
before_summary

,dtype,missing_values,non_missing,unique_values
age,int64,0,1338,47
sex,str,0,1338,2
bmi,float64,0,1338,548
children,int64,0,1338,6
smoker,str,0,1338,2
region,str,0,1338,4
charges,float64,0,1338,1337


** 1.5 And finally the clean dataset to will save as a clean file**

In [82]:
insurance.to_csv(r"C:\Users\skmra\Documents\healthcare_insurance_cost_analysis\insurance_cleaned.csv", index=False)

# Section 2

Clean Data analysis and Visualisation 

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [79]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)


IndentationError: expected an indented block after 'try' statement on line 2 (553063055.py, line 5)